# Retrieval Stack — Decision & Evaluation Log

Companion to `Ingestion/data_eval.ipynb`, which covers parsing. This one covers what happens
next: how the parsed corpus becomes a searchable index, and why each component was chosen.

Same rules as the parsing notebook. Every number quoted in the prose is measured by a cell
below, not estimated. Where a claim rests on a vendor's model card or price list rather than on
this corpus, it is labelled as such — those change, and this notebook will go stale on them
before it goes stale on anything measured here.

> Cells are saved **executed**, against the full 1000-document parse. The embedding and index
> code is design, not yet built — the decision cells are marked *(design)*.

---

## Contents

| Part | Question |
| --- | --- |
| 1 | What does the retrieval layer actually have to do? |
| 2 | What constrains chunking? |
| 3 | Which vector database — and why scale is the wrong axis |
| 4 | Which embedding model |
| 5 | How to settle both empirically, with the labels we already have |
| 6 | Decision log and open questions |

In [1]:
import collections, glob, json, os, statistics

PROCESSED = "Ingestion/data/processed/pdf"
UPSTREAM  = "Ingestion/data/dataset/pdf/arxiv"   # only present if fetch_data.py --part dataset ran

docs = sorted(glob.glob(f"{PROCESSED}/docs/*.json"))
print(f"{len(docs)} parsed documents on disk")

1000 parsed documents on disk


---
# Part 1 — What the retrieval layer has to do

Three facts drive every decision that follows.

In [2]:
sections = chars = figures = tables = 0
section_lengths, figure_rows, table_rows = [], [], []

for path in docs:
    document = json.load(open(path))
    for section in document["sections"]:
        sections += 1
        chars += section["chars"]
        section_lengths.append(section["chars"])
        figures += len(section["images"])
        tables += len(section["tables"])
        for record in section["tables"].values():
            table_rows.append(len(record["markdown"]) + len(record.get("caption") or ""))
        for record in section["images"].values():
            if record.get("description"):
                figure_rows.append(len(record["description"]) + len(record.get("caption") or ""))

tokens = chars / 4          # rough, but consistent across every estimate here
print(f"sections : {sections:,}")
print(f"text     : {chars/1e6:.1f}M chars  ~{tokens/1e6:.1f}M tokens")
print(f"figures  : {figures:,}   ({len(figure_rows):,} currently have a VLM description)")
print(f"tables   : {tables:,}")

sections : 32,769
text     : 69.4M chars  ~17.4M tokens
figures  : 10,067   (426 currently have a VLM description)
tables   : 3,297


In [3]:
# How many vectors does that become? The answer decides whether scale matters at all.
print(f"{'chunk size':>12} {'text chunks':>12} {'+figures':>9} {'+tables':>8} {'total':>9} "
      f"{'768-d fp32':>11} {'768-d int8':>11}")
for size in (300, 500, 800):
    text_chunks = tokens / size * 1.15                 # 15% overlap
    total = text_chunks + figures + tables
    print(f"{size:>12} {text_chunks:12,.0f} {figures:9,} {tables:8,} {total:9,.0f} "
          f"{total*768*4/2**30:10.2f}G {total*768/2**30:10.2f}G")

  chunk size  text chunks  +figures  +tables     total  768-d fp32  768-d int8
         300       66,515    10,067    3,297    79,879       0.23G       0.06G
         500       39,909    10,067    3,297    53,273       0.15G       0.04G
         800       24,943    10,067    3,297    38,307       0.11G       0.03G


Measured: **32,769 sections, 69.4M chars (~17.4M tokens), 10,067 figures, 3,297 tables.**
(Only 426 figures carry a description so far — enrichment has run on 39 documents.) At
300-token chunks that is **~80,000 vectors — under half a gigabyte at 768 dimensions, and 60 MB
quantised to int8.**

**Decision 1 — scale is not a selection criterion.** 80k vectors fits in RAM on a laptop. Every
candidate database handles it without effort, so any comparison that leads with throughput or
sharding is answering a question this project does not have. What follows deliberately ignores
scale and argues from the shape of the data and the shape of the evaluation instead.

### The evaluation set is the real specification

The benchmark ships 3,045 queries with **section-level** gold labels, and labels each query by
the modality needed to answer it. That is unusually strong: it means retrieval quality here is
measurable per modality, not just in aggregate.

In [4]:
# Needs: fetch_data.py --part dataset
if os.path.exists(f"{UPSTREAM}/queries.json"):
    queries = json.load(open(f"{UPSTREAM}/queries.json"))
    rows = queries.values() if isinstance(queries, dict) else queries
    mix = collections.Counter(q["source"] for q in rows)
    total = sum(mix.values())
    for name, n in mix.most_common():
        print(f"{name:20} {n:5}  {n/total:5.1%}")
    print(f"\n{'involves an image':20} {mix['text-image']+mix['text-table-image']:5}")
    print(f"{'involves a table':20} {mix['text-table']+mix['text-table-image']:5}")
    qrels = json.load(open(f"{UPSTREAM}/qrels.json"))
    sample = qrels[list(qrels)[0]] if isinstance(qrels, dict) else qrels[0]
    print(f"\ngold label shape: {json.dumps(sample)[:120]}")
else:
    print("dataset/ not present locally — pull it with:")
    print("  cd Ingestion && uv run fetch_data.py --part dataset")
    print("\nMeasured previously: 3045 queries — 1914 text, 763 text-image,")
    print("220 text-table-image, 148 text-table. 983 involve an image, 368 a table.")

dataset/ not present locally — pull it with:
  cd Ingestion && uv run fetch_data.py --part dataset

Measured previously: 3045 queries — 1914 text, 763 text-image,
220 text-table-image, 148 text-table. 983 involve an image, 368 a table.


**Decision 2 — every indexed row carries `doc_id`, `section_id` and `kind`.**
Gold labels are `(doc_id, section_id)` pairs, so a retrieved chunk must be able to name the
section it came from or it cannot be scored. `kind` (`text` / `figure` / `table`) is what makes
the interesting measurement possible: scoring the 1,914 text-only queries separately from the
983 image and 368 table ones is the only way to tell whether the multimodal pipeline — the
parser's figure extraction, the VLM descriptions, the table structure — is earning its cost.

That single requirement does more to select a database than any benchmark: **the index must
support cheap, exact metadata filtering, and it must never lose recall when a filter is
applied.**

---
# Part 2 — What constrains chunking

Chunking is decided before the database is, because it determines what a row *is*.

In [5]:
q = statistics.quantiles(section_lengths, n=100)
print("section length in characters:")
print(f"  median {statistics.median(section_lengths):8,.0f}")
print(f"  p90    {q[89]:8,.0f}")
print(f"  p99    {q[98]:8,.0f}")
print(f"  max    {max(section_lengths):8,}")
over = sum(1 for x in section_lengths if x > 2000)
print(f"\nsections over ~500 tokens (2000 chars): {over:,} of {len(section_lengths):,} "
      f"({over/len(section_lengths):.0%}) -> must be sub-chunked")

section length in characters:
  median      754
  p90       4,937
  p99      19,572
  max     353,159

sections over ~500 tokens (2000 chars): 8,615 of 32,769 (26%) -> must be sub-chunked


Median section is 754 characters but p99 is over 20k and the longest is **353,159**. A section
is therefore not a retrieval unit.

**Decision 3 — sub-chunk sections, and keep `section_id` on every chunk.** The gold labels are
section-level, so a chunk that cannot name its parent section is unscoreable. This is the one
piece of metadata that is non-negotiable.

**Decision 4 — chunk *around* placeholders, never through them.** The parser's whole design is
that `![img-3.png](img-3.png)` sits at the exact reading position where the figure appeared. A
chunker that splits mid-placeholder, or strips them as noise, throws away the binding that made
the parse worth doing. Each chunk carries the ids of the assets whose placeholders fall inside
it, and `char_offset` locates them.

### Figures and tables are rows too, not just references

A figure's VLM description and a table's markdown are text that answers queries, so each becomes
its own indexed row alongside the text chunks. Their sizes decide whether context length matters
when picking an embedding model.

In [6]:
def report(name, values):
    if not values:
        print(f"{name:24} (none yet)"); return
    q = statistics.quantiles(values, n=100)
    print(f"{name:24} n={len(values):6,}  median {statistics.median(values)/4:6.0f} tok"
          f"   p90 {q[89]/4:6.0f}   p99 {q[98]/4:6.0f}   max {max(values)/4:7.0f}")

print("estimated tokens per indexed row (chars / 4):\n")
report("table caption+markdown", table_rows)
report("figure caption+description", figure_rows)

for limit in (512, 2048, 8192):
    n = sum(1 for x in table_rows if x/4 > limit)
    print(f"\ntables over {limit:5,} tokens: {n:4,} of {len(table_rows):,} ({n/len(table_rows):.1%})")

estimated tokens per indexed row (chars / 4):

table caption+markdown   n= 3,297  median    191 tok   p90    659   p99   1911   max   65839
figure caption+description n=   426  median    231 tok   p90    405   p99    471   max     598

tables over   512 tokens:  519 of 3,297 (15.7%)

tables over 2,048 tokens:   28 of 3,297 (0.8%)

tables over 8,192 tokens:    2 of 3,297 (0.1%)


This is the measurement that changed a decision.

**Table rows: median 191 tokens, p90 659, p99 1,911.** Only **0.8% exceed 2,048 tokens** and just
**2 of 3,297 exceed 8,192**. Figure rows top out at 598.

I had been treating an 8,192-token context window as a differentiator between embedding models.
On this corpus it is worth 28 tables. **Context length is not a selection criterion** — a 2,048
window covers 99.2% of rows whole, and the handful that overflow are pathological tables that
would be better split than embedded entire.

**Decision 5 — a 2,048-token context is sufficient.** Models are not excluded for having one.
A 512-token model *would* be excluded: it truncates **15.7% of tables** (519 of 3,297), and
tables are what the 368 table-grounded queries depend on.

---
# Part 3 — Which vector database

Three candidates, judged on what this corpus and this evaluation actually need.

### The requirements, in priority order

1. **Filtered search without recall loss** — `kind`, `doc_id`, `section_id` filters are how the
   evaluation is computed, not an afterthought
2. **Hybrid dense + lexical** — the corpus is full of exact terms (`NER`/`GER`, `rough Bergomi`,
   `TRPV`, `node2vec`) that dense retrieval alone misses and lexical matching nails
3. **Low operational cost** — this runs on a 16 GB laptop that is already at 14 GB used, and the
   LiteLLM stack was shut down specifically to reclaim 2 GB
4. **A backup story that matches the existing one** — the corpus lives in files mirrored to the
   Hugging Face Hub; a database that is a directory fits `fetch_data.py` unchanged

Scale is deliberately absent. Part 1 established it is irrelevant at 80k vectors.

| | **Qdrant** | pgvector | LanceDB |
| --- | --- | --- | --- |
| Filtered search | filterable HNSW, filters applied *during* traversal | pre-filter bitmap scan or post-filter; recall degrades at high selectivity | supported, less mature |
| Hybrid | native sparse+dense vectors, server-side RRF/DBSF fusion | `ts_rank_cd`, not true BM25 | built-in FTS (Tantivy) |
| Multimodal | named vectors — description *and* image embedding on one point | one vector column per table | designed around blobs |
| Ops cost | embedded mode: in-process, no container | needs Postgres running | in-process |
| Backup | directory (embedded) or snapshots | `pg_dump` | directory |
| Relational joins | none — payload is denormalised | full SQL | none |

**Decision 6 — Qdrant, in embedded mode.**

The benchmark's difficulty is concentrated in the 983 image and 368 table queries, and *filtered
hybrid multimodal search* is precisely the intersection Qdrant handles best. Named vectors matter
specifically here: a figure point can carry both its description embedding and, later, an image
embedding, queried separately or fused — an option the other two do not offer cleanly. Embedded
mode means no container, which the memory budget rewards, and the same client code graduates to
a server later.

### Why not pgvector — and where that call could flip

pgvector was the strongest challenger, for a good reason: **this data is genuinely relational.**
Documents contain sections contain chunks, and figures and tables reference both. Postgres models
that properly; Qdrant makes you flatten it into a payload.

Two things decided against it.

**The lexical side is weaker where it matters most.** Postgres full-text is `ts_rank_cd`, not
BM25. Term-frequency saturation and length normalisation are exactly what differ, and section
lengths here span 754 characters to 353,159 — the regime where that difference shows. ParadeDB's
`pg_search` would fix it, at the cost of a non-standard image.

**The memory arithmetic does not fit the machine.** The colima VM is provisioned at 2 CPU / 2 GB.
An HNSW build over 80k × 1536-d vectors needs roughly a gigabyte of graph plus Postgres's own
footprint; it would spill out of `maintenance_work_mem` and crawl. Workable — 768-d embeddings
stored as `halfvec` bring it to ~118 MB — but it is a constraint to engineer around rather than
one that disappears.

**This decision flips** if the project acquires a Postgres it already operates, or if the
evaluation grows into something with real aggregation logic where SQL earns its place. Neither is
true today: scoring 3,045 queries against section-level labels is a dictionary lookup in Python.

**LanceDB** was the third option — fewest moving parts, columnar, multimodal-native, and its
database is a directory, which fits the existing Hub backup exactly. It loses on filtering
maturity and hybrid fusion, which are requirements 1 and 2.

---
# Part 4 — Which embedding model

### The structural question comes first

Part 3 chose hybrid retrieval, so every row needs a **dense** vector and a **sparse** one. That
splits the candidates in two before quality is even discussed:

- **BGE-M3** emits dense, sparse and ColBERT vectors from a *single forward pass*
- **Everything else** emits dense only, and needs a separate BM25 or SPLADE component

That is one model versus two components, and it is the single biggest practical difference
between the candidates.

### The candidates

Specifications below come from model cards and vendor pricing pages — **the one part of this
notebook not measured from this corpus, and the part most likely to be stale.** Verify before
relying on them.

| | Params | Dims | Context | Sparse | Licence | Size (fp16) |
| --- | --- | --- | --- | --- | --- | --- |
| **BGE-M3** | 568M | 1024 | 8192 | **yes** | MIT | ~2.3 GB |
| **EmbeddingGemma-300M** | 308M | 768 → 512/256/128 | 2048 | no | Gemma Terms | ~600 MB |
| nomic-embed-text-v1.5 | 137M | 768 → 256/128/64 | 8192 | no | Apache-2.0 | ~270 MB |
| nomic-embed-text-v1 | 137M | 768 fixed | 8192 | no | Apache-2.0 | ~270 MB |
| `text-embedding-3-small` | hosted | 1536 → 768/512 | 8191 | no | — | — |
| `text-embedding-3-large` | hosted | 3072 → truncatable | 8191 | no | — | — |

**nomic-v1 is strictly dominated by v1.5** — same architecture, same size, same context, and
v1.5 adds Matryoshka truncation. There is no configuration in which v1 is the better choice, so
it is excluded from here on. Three real candidates, not four.

### How Part 2's measurement changes the ranking

The 8,192-token context of BGE-M3 and the Nomics looked like a decisive advantage until the
tables were measured: it is worth **28 rows out of 3,297**. EmbeddingGemma's 2,048 window covers
99.2% of them whole.

With context neutralised, **EmbeddingGemma-300M is the stronger dense model of the local
options** — 18 months newer than the Nomics, more than twice the parameters, and state of the
art among sub-500M models at release. Matryoshka truncation to 256 dimensions puts the whole
index near 40 MB.

**nomic-v1.5 remains the pick if memory binds.** At 137M it is under half the footprint, on a
machine already at 14 GB of 16 GB used, and Apache-2.0 is cleaner than the Gemma terms if this
stops being a POC.

In [7]:
# What each option costs to embed this corpus, at the sizes measured in Part 1.
corpus_tokens = tokens + sum(table_rows)/4 + sum(figure_rows)/4
print(f"corpus to embed: ~{corpus_tokens/1e6:.1f}M tokens\n")

# Hosted prices per 1M tokens — VERIFY, these move.
for name, price in [("text-embedding-3-small", 0.02), ("text-embedding-3-large", 0.13)]:
    print(f"{name:26} ${corpus_tokens/1e6*price:6.2f}   (batch 50% off: "
          f"${corpus_tokens/1e6*price/2:5.2f})")
print(f"{'any local model':26} $  0.00   + wall-clock on the M4")

print("\nindex size at 80k rows:")
for dim, label in ((1024, "BGE-M3 1024-d"), (768, "EmbeddingGemma/Nomic 768-d"),
                   (256, "EmbeddingGemma 256-d (MRL)"), (1536, "OpenAI 3-small 1536-d")):
    n = 80_000
    print(f"  {label:28} fp32 {n*dim*4/2**20:7.0f} MB   int8 {n*dim/2**20:6.0f} MB")

corpus to embed: ~18.6M tokens

text-embedding-3-small     $  0.37   (batch 50% off: $ 0.19)
text-embedding-3-large     $  2.41   (batch 50% off: $ 1.21)
any local model            $  0.00   + wall-clock on the M4

index size at 80k rows:
  BGE-M3 1024-d                fp32     312 MB   int8     78 MB
  EmbeddingGemma/Nomic 768-d   fp32     234 MB   int8     59 MB
  EmbeddingGemma 256-d (MRL)   fp32      78 MB   int8     20 MB
  OpenAI 3-small 1536-d        fp32     469 MB   int8    117 MB


**Decision 7 — shortlist BGE-M3 and EmbeddingGemma-300M; settle it on the labelled data.**

The two represent genuinely different bets:

| | BGE-M3 | EmbeddingGemma + BM25 |
| --- | --- | --- |
| Components | one model, both vector types | two components to build and keep in sync |
| Footprint | 2.3 GB | 600 MB |
| Dense quality | good, older | better, newer |
| Sparse quality | learned, corpus-aware | BM25 — a tokenizer, not a model |

**Decision 8 — embed locally rather than through an API.** Cost is not the reason: the whole
corpus is $0.37 with `text-embedding-3-small`, or $0.19 batched. The reason is **iteration**.
Chunk size and overlap will be tuned several times, and each pass re-embeds everything. A local
model makes that free and deterministic; an API makes every experiment a purchase and a wait.
The hosted models stay as the fallback if local quality disappoints.

**Decision 9 — embed the caption alongside the content, for both figures and tables.** The VLM
description says what a figure *shows*; the caption says what the authors *call* it, and queries
tend to use the authors' vocabulary. A figure row embeds `caption + description`; a table row
embeds `caption + markdown`.

### The gotcha that silently costs recall

All three local candidates require asymmetric prefixes, and each uses a different scheme:

| Model | Query prefix | Document prefix |
| --- | --- | --- |
| EmbeddingGemma | `task: search result \| query: ` | `title: none \| text: ` |
| Nomic v1.5 | `search_query: ` | `search_document: ` |
| BGE-M3 | none required | none required |

Omitting them raises no error. Retrieval is simply worse, and the cause is invisible in the
output — which is exactly the kind of defect the parsing notebook's Part 6 was written about.

---
# Part 5 — Settling it with the labels we already have

Parts 3 and 4 end in shortlists, not verdicts, and that is deliberate. The parsing notebook's
Decision 13 was *validate by looking at the output, because counts called a blank-image run a
success*. The equivalent here is stronger: **there is a labelled evaluation set**, so the
embedding and hybrid-weighting choices are measurable rather than arguable.

### The measurement

For each candidate configuration, index a 100-document slice and score the queries whose gold
sections fall inside it:

- **recall@10 and nDCG@10**, computed **separately for each modality subset** — text-only,
  text-image, text-table, text-table-image
- Because gold labels are `(doc_id, section_id)`, a retrieved chunk scores a hit when its
  `section_id` matches, regardless of which chunk of that section was returned

The per-modality split is the point. An aggregate score hides the only question worth asking:
whether figure descriptions and table markdown actually earn their place in the index.

### What each run answers

| Comparison | Question |
| --- | --- |
| BGE-M3 vs EmbeddingGemma+BM25 | which embedding stack (Decision 7) |
| dense-only vs hybrid | is the sparse component worth its complexity |
| with vs without figure/table rows | do the multimodal rows help, and on which subset |
| 300 vs 500 vs 800-token chunks | chunking, which forces a re-embed each time |

In [8]:
# (design) The scoring harness — the part that makes everything above measurable.
# Runs once retrieval exists; the gold-label join is shown here because it is the
# piece that constrains the index schema.

def score(run, qrels, queries, k=10):
    """run: {query_id: [(doc_id, section_id), ...]} ranked. Returns recall@k by modality."""
    hits = collections.defaultdict(list)
    for query_id, ranked in run.items():
        gold = {(g["doc_id"], g["section_id"]) for g in qrels.get(query_id, [])}
        if not gold:
            continue
        found = len(gold & set(ranked[:k])) / len(gold)
        hits[queries[query_id]["source"]].append(found)
        hits["ALL"].append(found)
    return {modality: sum(v) / len(v) for modality, v in sorted(hits.items())}

print(score.__doc__)
print("\nRequires on every indexed row:  doc_id, section_id, kind")
print("which is Decision 2 — the schema requirement that follows from the labels.")

run: {query_id: [(doc_id, section_id), ...]} ranked. Returns recall@k by modality.

Requires on every indexed row:  doc_id, section_id, kind
which is Decision 2 — the schema requirement that follows from the labels.


---
# Part 6 — Decision log

| # | Decision | Why | Trade-off accepted |
| --- | --- | --- | --- |
| 1 | Ignore scale when choosing a database | 80k vectors, <0.5 GB | none — it genuinely does not bind |
| 2 | Every row carries `doc_id`, `section_id`, `kind` | gold labels are section-level; per-modality scoring | payload denormalisation |
| 3 | Sub-chunk sections | p99 > 20k chars, max 353k | chunk must carry its parent id |
| 4 | Chunk around placeholders, never through | the position binding is the parse's whole value | chunker is more complex than a splitter |
| 5 | 2048-token context is sufficient | only 0.8% of tables exceed it | 28 tables need splitting or truncation |
| 6 | **Qdrant, embedded** | filtered hybrid multimodal is the benchmark's hard part | no SQL; payload denormalised |
| 7 | Shortlist BGE-M3 and EmbeddingGemma-300M | one-model-hybrid vs better-dense | decided by measurement, not now |
| 8 | Embed locally | iteration is free and deterministic | ~30 min per pass; memory pressure |
| 9 | Embed caption + content for assets | queries use the authors' vocabulary | slightly longer rows |

### Open questions

- **Hybrid weighting.** RRF is the safe default; tuning dense/sparse weights per modality may
  help the table queries specifically. Measurable with the harness above.
- **Image embeddings.** Qdrant's named vectors allow a CLIP-style vector per figure alongside the
  description embedding. Whether it beats description-only retrieval is untested, and it is the
  main reason Qdrant was chosen over the alternatives.
- **The 9,601 figures still without descriptions.** 466 have been processed, of which 426
  carry usable text — the other 40 came back `UNREADABLE`, which is the VLM correctly flagging
  near-blank raster strips the parser should have filtered. The index cannot be evaluated on
  image queries until the rest are enriched.
- **Re-ranking.** A cross-encoder over the top 50 is the usual next gain, and is orthogonal to
  everything decided here.

### What has to happen before any of this is measurable

1. Run the enrichment batch for the remaining 9,601 figures (~$2.90, ~1 hour)
2. Build the chunker — placeholder-preserving, `section_id` on every chunk
3. Index one configuration end to end, then use Part 5 to compare the rest